In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys, os, re
from pathlib import Path
from typing import Optional
from scipy.signal import savgol_filter
from sklearn.preprocessing import normalize
sys.path.append('../')

from src import *

seed = 42
np.random.seed(seed)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
df = pd.read_parquet("../data/all_raman_spectra.parquet")
df.head()

,X,Y,Wave,Intensity,label,group,center,brain,place
0,-511.945953,-16787.546875,2002.376953,13577.683594,exo,2b,1500,striatum_right,2_4
1,-511.945953,-16787.546875,2001.416992,13663.965820,exo,2b,1500,striatum_right,2_4
2,-511.945953,-16787.546875,2000.457031,13875.706055,exo,2b,1500,striatum_right,2_4
3,-511.945953,-16787.546875,1999.496094,13786.798828,exo,2b,1500,striatum_right,2_4
4,-511.945953,-16787.546875,1998.536133,13700.540039,exo,2b,1500,striatum_right,2_4


In [ ]:
window_length = 5 if len(df) >= 5 else len(df) | 1
polyorder = 2

df = df.sort_values(['X','Y','Wave'])

df['Intensity_smooth'] = (
    df.groupby(['X','Y'])['Intensity']
      .transform(
          lambda x: savgol_filter(
              x,
              min(window_length if window_length % 2 else window_length+1, len(x)-(len(x)+1)%2),
              polyorder
          )
      )
)

df['Intensity_normalized'] = (
    df.groupby(['X','Y'])['Intensity_smooth']
      .transform(lambda x: normalize(x.values.reshape(1,-1), norm='l1')[0])
)

df['Intensity_derivative'] = (
    df.groupby(['X','Y'], group_keys=False)
      .apply(lambda g: pd.Series(
          np.gradient(g['Intensity_normalized'], g['Wave']),
          index=g.index
      ))
)

df.head()

C:\Users\gmart\AppData\Local\Temp\ipykernel_15856\3415810486.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series(


,X,Y,Wave,Intensity,label,group,center,brain,place,Intensity_smooth,Intensity_normalized,Intensity_derivative
29309139,-23710.740234,-2048.758301,926.803711,8912.721680,exo,1,1500,cortex,6_1,8888.870117,0.000520,2.813032e-06
29309138,-23710.740234,-2048.758301,927.976562,8912.397461,exo,1,1500,cortex,6_1,8945.297852,0.000523,2.484681e-06
29309137,-23710.740234,-2048.758301,929.150391,8944.176758,exo,1,1500,cortex,6_1,8988.583008,0.000526,2.648259e-06
29309136,-23710.740234,-2048.758301,930.323242,9110.833984,exo,1,1500,cortex,6_1,9051.571289,0.000529,1.949520e-07
29309135,-23710.740234,-2048.758301,931.495117,8997.069336,exo,1,1500,cortex,6_1,8996.499023,0.000526,-2.857269e-06


In [5]:
file_path = "../data/processed_dataset.parquet"

if not os.path.exists(file_path):
    df.to_parquet(file_path, engine='pyarrow', index=False)
    print(f"Преобразованные данные: {file_path}")
else:
    print(f"Файл с преобразованными данными:", file_path)

Файл с преобразованными данными: ../data/processed_dataset.parquet


# ML Baseline (XGBoost)

In [6]:
frame = df.sample(n=1000000, random_state=42)
frame.info()

frame['label'].value_counts()
frame['label'] = frame['label'].map({
    'control': 0,
    'exo': 1,
    'endo': 2
})

<class 'pandas.core.frame.DataFrame'>
Index: 1000000 entries, 120458098 to 104867774
Data columns (total 12 columns):
 #   Column                Non-Null Count    Dtype   
---  ------                --------------    -----   
 0   X                     1000000 non-null  float32 
 1   Y                     1000000 non-null  float32 
 2   Wave                  1000000 non-null  float32 
 3   Intensity             1000000 non-null  float32 
 4   label                 1000000 non-null  category
 5   group                 1000000 non-null  category
 6   center                1000000 non-null  category
 7   brain                 1000000 non-null  category
 8   place                 1000000 non-null  category
 9   Intensity_smooth      1000000 non-null  float32 
 10  Intensity_normalized  1000000 non-null  float32 
 11  Intensity_derivative  1000000 non-null  float32 
dtypes: category(5), float32(7)
memory usage: 39.1 MB


In [7]:
from xgboost import XGBClassifier 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC

In [8]:
features = ['X', 'Y', 'Wave', 'Intensity', 'Intensity_smooth', 'Intensity_normalized', 'Intensity_derivative', 'center']
X = frame[features]
y = frame['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    ))
])

In [10]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [11]:
# print(f'F1-Score: {f1_score(y_test, y_pred)}')
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(f"Precision = {precision_score(y_test, y_pred, average='macro')} | Recall = {recall_score(y_test, y_pred, average='macro')}")
print(f'Classification report:\n{classification_report(y_test, y_pred)}')

Accuracy: 0.99992
Precision = 0.9999146998100827 | Recall = 0.9999233852168089
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     67575
           1       1.00      1.00      1.00     70955
           2       1.00      1.00      1.00     61470

    accuracy                           1.00    200000
   macro avg       1.00      1.00      1.00    200000
weighted avg       1.00      1.00      1.00    200000

